# Privacy Guardian v2 - NER training (M1)
Fine-tunes `distilbert-base-cased` on ai4privacy open-pii-masking-500k (English). See ADR-0009, 0010, 0011.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.

In [ ]:
!pip install -q transformers datasets accelerate seqeval
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4!")

## 1. Get our label code from the repo
The repo must be public for this clone. If it is private, upload `backend/app` as a zip and unzip it to `/content/privacy-guardian/backend`.

In [ ]:
!rm -rf privacy-guardian && git clone -q https://github.com/skiing22/privacy-guardian.git
import sys
sys.path.insert(0, "/content/privacy-guardian/backend")
from app.detectors.label_map import map_spans
from app.detectors.ner_labels import IGNORE, align_labels, build_label_list, span_scores

LABELS = build_label_list()
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(len(LABELS), "labels:", LABELS)

## 2. Settings

In [ ]:
MODEL_NAME = "distilbert-base-cased"   # cased: capital letters help find names (ADR-0011)
N_TRAIN = 50_000
N_VAL = 5_000
MAX_LEN = 256
EPOCHS = 2
BATCH = 32
LR = 5e-5
SEED = 42

## 3. Load data and map labels

In [ ]:
import json
from datasets import load_dataset

ds = load_dataset("ai4privacy/open-pii-masking-500k-ai4privacy")
en_train = ds["train"].filter(lambda r: r["language"] == "en").shuffle(seed=SEED)
en_val = ds["validation"].filter(lambda r: r["language"] == "en").shuffle(seed=SEED)
train_raw = en_train.select(range(min(N_TRAIN, len(en_train))))
val_raw = en_val.select(range(min(N_VAL, len(en_val))))
print(len(en_train), "English train rows available; using", len(train_raw), "/", len(val_raw))

def gold_spans(row):
    mask = row["privacy_mask"]
    mask = json.loads(mask) if isinstance(mask, str) else mask
    raw = [(int(m["start"]), int(m["end"]), m["label"]) for m in mask]
    return map_spans(row["source_text"], raw)

ex = train_raw[0]
print(ex["source_text"][:200])
print([(ex["source_text"][s:e], t.value) for s, e, t in gold_spans(ex)])

## 4. Tokenize and align labels

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    enc = tok(batch["source_text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    all_labels = []
    for i in range(len(batch["source_text"])):
        row = {"source_text": batch["source_text"][i], "privacy_mask": batch["privacy_mask"][i]}
        all_labels.append(align_labels(enc["offset_mapping"][i], enc.word_ids(i), gold_spans(row), label2id))
    enc["labels"] = all_labels
    enc.pop("offset_mapping")
    return enc

cols = train_raw.column_names
train_ds = train_raw.map(encode, batched=True, remove_columns=cols)
val_ds = val_raw.map(encode, batched=True, remove_columns=cols)
print(train_ds)

## 5. Train

In [ ]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score
from transformers import (AutoModelForTokenClassification, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    y_true, y_pred = [], []
    for pr, lb in zip(preds, p.label_ids):
        keep = lb != IGNORE
        y_true.append([id2label[int(x)] for x in lb[keep]])
        y_pred.append([id2label[int(x)] for x in pr[keep]])
    return {"precision": precision_score(y_true, y_pred), "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred)}

import math
total_steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
args = TrainingArguments(
    output_dir="ner-out", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
    weight_decay=0.01, warmup_steps=int(0.1 * total_steps),  # 10% warmup; works on v4 and v5
    eval_strategy="epoch", save_strategy="no",
    logging_steps=200, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  processing_class=tok, data_collator=DataCollatorForTokenClassification(tok),
                  compute_metrics=compute_metrics)
trainer.train()
token_metrics = trainer.evaluate()
print(token_metrics)

## 6. Real test: exact span match (start, end and type must all be right)
This is the number that goes in the README, not the token score above.

In [ ]:
from transformers import pipeline
ner = pipeline("token-classification", model=trainer.model, tokenizer=tok,
               aggregation_strategy="first", device=0 if torch.cuda.is_available() else -1)

texts = val_raw["source_text"]
preds = ner(list(texts), batch_size=64)

def strip(text, s, e):
    while s < e and text[s].isspace(): s += 1
    while e > s and text[e - 1].isspace(): e -= 1
    return s, e

gold, pred = [], []
for i, text in enumerate(texts):
    row = val_raw[i]
    gold.append({(s, e, t.value) for s, e, t in gold_spans(row)})
    pred.append({(*strip(text, p["start"], p["end"]), p["entity_group"]) for p in preds[i]})

scores = span_scores(gold, pred)
print(f"{'type':<10}{'P':>7}{'R':>7}{'F1':>7}{'n':>7}")
for t, m in scores.items():
    print(f"{t:<10}{m['precision']:7.3f}{m['recall']:7.3f}{m['f1']:7.3f}{int(m['support']):7d}")

## 7. Try Indian-style text
The data has few Indian names, so check by eye.

In [ ]:
samples = [
    "Hi, I am Priya Venkataraman from Kozhikode. Call me on 98470 12345 or mail priya.v@gmail.com.",
    "Patient Arjun Nair, age 34, admitted on 12/03/2026. Address: 14 MG Road, Kochi 682016.",
]
for s in samples:
    print(s)
    for p in ner(s):
        print(f"   {p['entity_group']:<9} {p['score']:.2f}  {s[p['start']:p['end']]!r}")

## 8. Save to Google Drive
Model weights never go into git (ADR-0006). Download the folder later for the backend.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/privacy-guardian/ner-distilbert-v1"
trainer.save_model(OUT)
tok.save_pretrained(OUT)
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump({"token_level": token_metrics, "span_level": scores,
               "config": {"model": MODEL_NAME, "n_train": len(train_raw), "n_val": len(val_raw),
                          "epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN}}, f, indent=2)
print("saved to", OUT)